In [ ]:
import core.dataset
import core.experiment
import core.grid
import numpy as np
from scripts.analyze_mlflow_runs import (
    analyze_unified_mlruns,
    aggregate_metrics,
    print_summary,
    plot_metrics,
)
from helpers import utils
from scripts import diagnose_fold_differences as fold_dx
import re
import os
from collections import defaultdict
from reload_recursive import reload_recursive
from pathlib import Path
from IPython.display import clear_output
from tqdm.notebook import tqdm
import pandas as pd

In [54]:
reload_recursive(core.dataset)
from core.dataset import Dataset

reload_recursive(core.experiment)
from core.experiment import Experiment

reload_recursive(core.grid)
from core.grid import ExperimentGrid

reload_recursive(core.configs)
from core.configs import AlgoConfig

reload_recursive(utils)


In [72]:
import sys
from loguru import logger

# Map module names (or substrings) to minimum log levels.
# Anything not listed gets the default level.
MODULE_LEVELS = {
    "generate_fold_predictions": "WARNING",  # quiet during predict()
    "inference": "INFO",
    "grid": "DEBUG",
}
DEFAULT_LEVEL = "INFO"


def _module_filter(record):
    name = record["name"]  # e.g. "scripts.generate_fold_predictions"
    for module, level in MODULE_LEVELS.items():
        if module in name:
            return record["level"].no >= logger.level(level).no
    return record["level"].no >= logger.level(DEFAULT_LEVEL).no


logger.remove()  # drop the default stderr handler
logger.add(
    sys.stderr,
    filter=_module_filter,
    format="{time:HH:mm:ss} | {level:<7} | {name} - {message}",
)

MODULE_LEVELS["generate_fold_predictions"] = "INFO"
logger.remove()
logger.add(sys.stderr, filter=_module_filter)

12

In [ ]:
train_root = Path(os.environ["PRL_TRAIN_ROOT"])


def load_grid(ds_name, grid_name):
    config = train_root / ds_name / grid_name / "experiment_config.yaml"
    return ExperimentGrid.from_config(config)

In [73]:
gridsearch_experiments = [
    "stage4_sweep_dicece_wts",
    "stage5_sweep_dicecewt_nbatch",
    "stage3_numcrops_bkd_constwt115",
    "stage1_crop_lr_sweep",
    "stage2_numcrops_dicece",
]
params_to_gather = [
    "learning_rate",
    "crop_ratios",
    "num_crops_per_image",
    "batch_size",
    "loss#weight",
    "loss#include_background",
]

In [74]:
dataset_name = "roi_train2"
for gridsearch_name in gridsearch_experiments:
    # gridsearch_name = "stage5_sweep_dicecewt_nbatch"
    grid = load_grid(dataset_name, gridsearch_name)
    run_params = grid.run_params(*grid.get_info())
    ds = Dataset(dataset_name)
    runs_to_check = ["run1"]
    runs = []
    for run_info in run_params:
        run_loc = f"{grid.experiment_name}/{run_info['run_name']}"
        print(f"STARTING {run_loc}")
        exp = Experiment.from_run_dir(run_loc, ds)
        exp.predict()
        exp.cases #builds the case list for the first time and caches for quick access
        runs.append({**run_info, "exp": exp})

# clear_output()

2026-03-25 02:00:00.584 | INFO     | scripts.generate_fold_predictions:run_fold_inference:57 - Running inference for fold 0


STARTING stage4_sweep_dicece_wts/run1


<frozen importlib._bootstrap_external>:1324: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
monai.metrics.meandice DiceHelper.__init__:sigmoid: Argument `sigmoid` has been deprecated since version 1.5. It will be removed in version 1.7. Use `threshold` instead.
monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will res

STARTING stage4_sweep_dicece_wts/run2


<frozen importlib._bootstrap_external>:1324: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
monai.metrics.meandice DiceHelper.__init__:sigmoid: Argument `sigmoid` has been deprecated since version 1.5. It will be removed in version 1.7. Use `threshold` instead.
monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will res

...

2026-03-25 02:32:39.188 | INFO     | scripts.generate_fold_predictions:run_fold_inference:73 - Fold 0: outputs already exist (159 files), skipping
2026-03-25 02:32:39.189 | INFO     | scripts.generate_fold_predictions:run_fold_inference:57 - Running inference for fold 1
2026-03-25 02:32:41.194 | INFO     | scripts.generate_fold_predictions:run_fold_inference:73 - Fold 1: outputs already exist (159 files), skipping
2026-03-25 02:32:41.195 | INFO     | scripts.generate_fold_predictions:run_fold_inference:57 - Running inference for fold 2
2026-03-25 02:32:43.125 | INFO     | scripts.generate_fold_predictions:run_fold_inference:73 - Fold 2: outputs already exist (159 files), skipping
2026-03-25 02:32:43.126 | INFO     | scripts.generate_fold_predictions:run_fold_inference:57 - Running inference for fold 3
2026-03-25 02:32:44.977 | INFO     | scripts.generate_fold_predictions:run_fold_inference:73 - Fold 3: outputs already exist (158 files), skipping
2026-03-25 02:32:44.978 | INFO     | scr

STARTING stage2_numcrops_dicece/run3


2026-03-25 02:32:53.270 | INFO     | scripts.generate_fold_predictions:run_fold_inference:73 - Fold 0: outputs already exist (159 files), skipping
2026-03-25 02:32:53.271 | INFO     | scripts.generate_fold_predictions:run_fold_inference:57 - Running inference for fold 1
2026-03-25 02:32:55.205 | INFO     | scripts.generate_fold_predictions:run_fold_inference:73 - Fold 1: outputs already exist (159 files), skipping
2026-03-25 02:32:55.205 | INFO     | scripts.generate_fold_predictions:run_fold_inference:57 - Running inference for fold 2
2026-03-25 02:32:57.043 | INFO     | scripts.generate_fold_predictions:run_fold_inference:73 - Fold 2: outputs already exist (159 files), skipping
2026-03-25 02:32:57.044 | INFO     | scripts.generate_fold_predictions:run_fold_inference:57 - Running inference for fold 3
2026-03-25 02:32:58.898 | INFO     | scripts.generate_fold_predictions:run_fold_inference:73 - Fold 3: outputs already exist (158 files), skipping
2026-03-25 02:32:58.899 | INFO     | scr

STARTING stage2_numcrops_dicece/run4


2026-03-25 02:33:07.299 | INFO     | scripts.generate_fold_predictions:run_fold_inference:73 - Fold 0: outputs already exist (159 files), skipping
2026-03-25 02:33:07.300 | INFO     | scripts.generate_fold_predictions:run_fold_inference:57 - Running inference for fold 1
2026-03-25 02:33:09.222 | INFO     | scripts.generate_fold_predictions:run_fold_inference:73 - Fold 1: outputs already exist (159 files), skipping
2026-03-25 02:33:09.222 | INFO     | scripts.generate_fold_predictions:run_fold_inference:57 - Running inference for fold 2
2026-03-25 02:33:11.286 | INFO     | scripts.generate_fold_predictions:run_fold_inference:73 - Fold 2: outputs already exist (159 files), skipping
2026-03-25 02:33:11.286 | INFO     | scripts.generate_fold_predictions:run_fold_inference:57 - Running inference for fold 3
2026-03-25 02:33:13.263 | INFO     | scripts.generate_fold_predictions:run_fold_inference:73 - Fold 3: outputs already exist (158 files), skipping
2026-03-25 02:33:13.263 | INFO     | scr

In [ ]:
# should return a str like "$torch.tensor([1.0, 3.0, 5.0]).cuda()"
wt_str = runs[0]["exp"].hyper_params["loss#weight"] 
# The tensor can be made to be nicer for analysis tables with
str_tensor = re.match(r"\$torch\.tensor\(\[(.+)\]\).+", wt_str)[1]
weights = [float(w) for w in str_tensor.split(",")]


# should return a boolean
runs[0]["exp"].hyper_params["loss#include_background"] 

# this is an old function I wrote, im not using it anymore, but its what
def get_loss_param(self, param):
    import re
    val = self.loss.get(param, None)
    if isinstance(val, str) and "torch" in val:
        str_tensor = re.match(r"\$torch\.tensor\(\[(.+)\]\).+", val)[1]
        return [float(w) for w in str_tensor.split(",")]
    else:
        return val

1

In [ ]:
gridsearch_name = "stage5_sweep_dicecewt_nbatch"
# I had written the function get_info() in grid.py to get the grid parameters
# in a more table friendly format, but it depended on the user specifying table friendly
#  keys within the experiment.yaml definition. Instead, lets just define a mapping for each key and type of value
#   in one spot as a function or class. It can be a growing function as I try to test more hyperparameters in the future
run_params = grid.run_params(*grid.get_info())

ds = Dataset(dataset_name)
runs_to_check = ["run1"]
runs = []
for run_info in run_params:
    run_loc = f"{grid.experiment_name}/{run_info['run_name']}"
    print(f"STARTING {run_loc}")
    exp = Experiment.from_run_dir(run_loc, ds)
    exp.predict()
    exp.cases #builds the case list for the first time and caches for quick access
    runs.append({**run_info, "exp": exp})


def compile_metrics(runs):
    compiled_metrics = defaultdict(list)
    for run in tqdm(runs, total=len(runs)):
        compiled_metrics["run"].append(run["run_name"])
        for hyperparam_key, hyperparam_val in run["training"].items():
            compiled_metrics[hyperparam_key].append(hyperparam_val)
        fold_data = analyze_unified_mlruns(run["exp"].run_dir)
        aggregated = aggregate_metrics(fold_data)
        by_prefix = defaultdict(dict)
        for metric_name in aggregated:
            if "/" in metric_name:
                prefix = metric_name.split("/")[0]
                by_prefix[prefix][metric_name] = aggregated[metric_name]
            else:
                by_prefix["other"][metric_name] = aggregated[metric_name]

        for prefix in sorted(by_prefix.keys()):
            for metric_name in sorted(by_prefix[prefix].keys()):
                metric_data = aggregated[metric_name]

                if "stats" not in metric_data:
                    continue

                stats = metric_data["stats"]
                for stat in ["mean", "std", "min", "max"]:
                    compiled_metrics[f"{metric_name}_{stat}"].append(
                        np.round(stats[stat], 4)
                    )

                fold_values = [
                    (fold_num, metric_data[fold_num])
                    for fold_num in sorted(fold_data.keys())
                    if fold_num in metric_data
                ]
                if "val_class" not in metric_name:
                    continue
                for fold_num, values in fold_values:
                    compiled_metrics[f"fold{fold_num}-{metric_name}_final"].append(
                        np.round(values[-1], 4)
                    )
                    compiled_metrics[f"fold{fold_num}-{metric_name}_max"].append(
                        np.round(np.max(values), 4)
                    )

    metrics = pd.DataFrame(compiled_metrics).set_index("run")
    col_remap = {}
    for col in metrics.columns:
        # lesion accs
        new_col = re.sub(r"^(.*)(val_)(class/acc_0)(_.+)", r"\1lesion/acc\4", col)
        col_remap[col] = new_col
        # rim accs
        col_remap[col] = re.sub(
            r"^(.*)(val_)(class/acc_1)(_.+)", r"\1rim/acc\4", new_col
        )

    metrics = metrics.rename(columns=col_remap)
    col_order_cats = defaultdict(list)
    for col in metrics.columns:
        if "acc_min" in col or "loss_max" in col:
            continue
        if "rim" in col:
            col_order_cats["rim"].append(col)
        elif "lesion" in col:
            col_order_cats["lesion"].append(col)
        elif "train" in col:
            col_order_cats["loss"].append(col)
        elif "val" in col:
            col_order_cats["val"].append(col)
        else:
            col_order_cats["run_info"].append(col)

    column_order = []
    for cat in ["run_info", "rim", "lesion", "loss", "val"]:
        column_order.extend(col_order_cats[cat])
    metrics = metrics[column_order]

    return metrics


  0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
metrics = compile_metrics(runs_to_analyze)
metrics.to_csv(f"{gridsearch_name}_metrics.csv")

In [ ]:
import pandas as pd

metrics = pd.DataFrame(compiled_metrics).set_index("run")
col_remap = {}
for col in metrics.columns:
    # lesion accs
    new_col = re.sub(r"^(.*)(val_)(class/acc_0)(_.+)", r"\1lesion/acc\4", col)
    col_remap[col] = new_col
    # rim accs
    col_remap[col] = re.sub(r"^(.*)(val_)(class/acc_1)(_.+)", r"\1rim/acc\4", new_col)

metrics = metrics.rename(columns=col_remap)
col_order_cats = defaultdict(list)
for col in metrics.columns:
    if "acc_min" in col or "loss_max" in col:
        continue
    if "rim" in col:
        col_order_cats["rim"].append(col)
    elif "lesion" in col:
        col_order_cats["lesion"].append(col)
    elif "train" in col:
        col_order_cats["loss"].append(col)
    elif "val" in col:
        col_order_cats["val"].append(col)
    else:
        col_order_cats["run_info"].append(col)

column_order = []
for cat in ["run_info", "rim", "lesion", "loss", "val"]:
    column_order.extend(col_order_cats[cat])
metrics = metrics[column_order]
metrics.to_csv(f"{gridsearch_name}_metrics.csv")

In [ ]:
metrics2 = metrics.copy()
metrics2.insert(2, "rim_dice_mean (PRL test cases)", None)
metrics2.insert(3, "rim_dice_std (PRL test cases)", None)
metrics2.insert(4, "lesion_dice_mean (PRL test cases)", None)
metrics2.insert(5, "lesion_dice_std (PRL test cases)", None)

for run in tqdm(runs, total=len(runs)):
    exp = run["exp"]
    testing_cases = [
        item
        for item in exp.cases
        if item["split"] == "testing" and item["case_type"] == "PRL"
    ]
    rim_dice = []
    lesion_dice = []
    for item in testing_cases:
        rim_dice.append(
            utils.dice_score(item["label"], item["inference"], seg1_val=2, seg2_val=2)
        )
        lesion_dice.append(
            utils.dice_score(item["label"], item["inference"], seg1_val=1, seg2_val=1)
        )
    metrics2.loc[run["run_name"], "rim_dice_mean (PRL test cases)"] = np.mean(rim_dice)
    metrics2.loc[run["run_name"], "rim_dice_std (PRL test cases)"] = np.std(rim_dice)
    metrics2.loc[run["run_name"], "lesion_dice_mean (PRL test cases)"] = np.mean(
        lesion_dice
    )
    metrics2.loc[run["run_name"], "lesion_dice_std (PRL test cases)"] = np.std(
        lesion_dice
    )
metrics.to_csv(f"{gridsearch_name}_metrics_w_dice.csv")

  0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
compiled_metrics = defaultdict(list)
for run in runs:
    compiled_metrics["run"].append(run["run_name"])
    compiled_metrics["lr"].append(exp.training_config.learning_rate)
    crop_ratios = exp.training_config.crop_ratios
    if crop_ratios is not None:
        compiled_metrics["crop_ratios"].append(",".join([str(x) for x in crop_ratios]))
    else:
        compiled_metrics["crop_ratios"].append("null")
    fold_data = analyze_unified_mlruns(exp.run_dir)
    aggregated = aggregate_metrics(fold_data)
    by_prefix = defaultdict(dict)
    for metric_name in aggregated:
        if "/" in metric_name:
            prefix = metric_name.split("/")[0]
            by_prefix[prefix][metric_name] = aggregated[metric_name]
        else:
            by_prefix["other"][metric_name] = aggregated[metric_name]

    for prefix in sorted(by_prefix.keys()):
        for metric_name in sorted(by_prefix[prefix].keys()):
            metric_data = aggregated[metric_name]

            if "stats" not in metric_data:
                continue

            stats = metric_data["stats"]
            for stat in ["mean", "std", "min", "max"]:
                compiled_metrics[f"{metric_name}_{stat}"].append(stats[stat])

            fold_values = [
                (fold_num, metric_data[fold_num])
                for fold_num in sorted(fold_data.keys())
                if fold_num in metric_data
            ]
            if "val_class" not in metric_name:
                continue
            for fold_num, values in fold_values:
                compiled_metrics[f"fold{fold_num}-{metric_name}_final"].append(
                    values[-1]
                )
                compiled_metrics[f"fold{fold_num}-{metric_name}_max"].append(
                    np.max(values)
                )

In [54]:
metrics2.to_csv("gridsearch_metrics_w_dice.csv")

In [ ]:
run_dir = runs[1].run_dir
datastats = fold_dx.load_datastats(run_dir)
rows = []
for item in runs[1].cases:
    if item["split"] == "testing":
        continue
    rows.append({k: item[k] for k in ["subid", "lesion_index", "split", "case_type"]})
metadata_df = pd.DataFrame(rows)
df = metadata_df.merge(datastats, on=["subid", "lesion_index"], how="left")

Loading /media/smbshare/srs-9/prl_project/training/roi_train2/stage1_crop_lr_sweep/run1/datastats_by_case.yaml ...
  Loaded 794 cases from datastats


In [ ]:
all_numerical_cols = df.select_dtypes(include="number").columns
ignore_numerical_cols = ["subid", "lesion_index"]
cols_to_mean = all_numerical_cols[~all_numerical_cols.isin(ignore_numerical_cols)]
means_by_fold = df.groupby("split")[cols_to_mean].mean()

In [55]:
means_by_fold.to_csv("fold_image_stats.csv")

In [ ]:
col_remap = {
    "val_class/acc_0_mean": "val_lesion/acc_mean",
    "val_class/acc_0_std": "val_lesion/acc_std",
    "val_class/acc_0_min": "val_lesion/acc_min",
    "val_class/acc_0_max": "val_lesion/acc_max",
    "val_class/acc_1_mean": "val_rim/acc_mean",
    "val_class/acc_1_std": "val_rim/acc_std",
    "val_class/acc_1_min": "val_rim/acc_min",
    "val_class/acc_1_max": "val_rim/acc_max",
}
col_order = [
    "lr",
    "crop_ratios",
    "val_rim/acc_mean",
    "val_rim/acc_max",
    "val_rim/acc_std",
    "val_lesion/acc_mean",
    "val_lesion/acc_max",
    "val_lesion/acc_std",
    "train/loss_mean",
    "train/loss_min",
    "train/loss_std",
    "val/acc_mean",
    "val/acc_max",
    "val/acc_std",
]
metrics = metrics.rename(columns=col_remap)[col_order]


In [62]:
round_columns = metrics.columns[~metrics.columns.isin(["lr", "run", "crop_ratios"])]
metrics[round_columns] = metrics[round_columns].round(3)

In [ ]:
col_remap = {
    "train/loss_mean",
    "train/loss_std",
    "train/loss_min",
    "train/loss_max",
    "val/acc_mean",
    "val/acc_std",
    "val/acc_min",
    "val/acc_max",
    "val_lesion/acc_mean",
    "val_lesion/acc_std",
    "val_lesion/acc_min",
    "val_lesion/acc_max",
    "val_rim/acc_mean",
    "val_rim/acc_std",
    "val_rim/acc_min",
    "val_rim/acc_max",
}

Index(['lr', 'crop_ratios', 'train/loss_mean', 'train/loss_std',
       'train/loss_min', 'train/loss_max', 'val/acc_mean', 'val/acc_std',
       'val/acc_min', 'val/acc_max', 'val_class/acc_0_mean',
       'val_class/acc_0_std', 'val_class/acc_0_min', 'val_class/acc_0_max',
       'val_class/acc_1_mean', 'val_class/acc_1_std', 'val_class/acc_1_min',
       'val_class/acc_1_max'],
      dtype='object')

In [40]:
performance_metrics = runs[1].evaluate()

/media/smbshare/srs-9/prl_project/data/sub2131-20190117/26/lesion_xy20_z2.nii.gz /media/smbshare/srs-9/prl_project/training/roi_train2/stage1_crop_lr_sweep/run1/ensemble_output/sub2131-20190117/26/flair.phase_xy20_z2_ensemble.nii.gz
/media/smbshare/srs-9/prl_project/data/sub1010-20180208/27/lesion_xy20_z2.nii.gz /media/smbshare/srs-9/prl_project/training/roi_train2/stage1_crop_lr_sweep/run1/ensemble_output/sub1010-20180208/27/flair.phase_xy20_z2_ensemble.nii.gz
/media/smbshare/srs-9/prl_project/data/sub1293-20161129/70/lesion_xy20_z2.nii.gz /media/smbshare/srs-9/prl_project/training/roi_train2/stage1_crop_lr_sweep/run1/ensemble_output/sub1293-20161129/70/flair.phase_xy20_z2_ensemble.nii.gz
194 valid cases of 197
/media/smbshare/srs-9/prl_project/data/sub1050-20170624/86/lesion_xy20_z2.nii.gz /media/smbshare/srs-9/prl_project/training/roi_train2/stage1_crop_lr_sweep/run1/fold_predictions/fold0/sub1050-20170624/86/flair.phase_xy20_z2.nii.gz
/media/smbshare/srs-9/prl_project/data/sub1050-

In [42]:
runs[1].cases[0]

{'subid': 1293,
 'lesion_index': 1,
 'image': PosixPath('/media/smbshare/srs-9/prl_project/data/sub1293-20161129/1/flair.phase_xy20_z2.nii.gz'),
 'label': PosixPath('/media/smbshare/srs-9/prl_project/data/sub1293-20161129/1/prl_label_SRS_CH_xy20_z2.nii.gz'),
 'split': 'testing',
 'inference': PosixPath('/media/smbshare/srs-9/prl_project/training/roi_train2/stage1_crop_lr_sweep/run1/ensemble_output/sub1293-20161129/1/flair.phase_xy20_z2_ensemble.nii.gz')}

In [41]:
performance_metrics

,subid,lesion_index,split,case_type,lesion_dice,prl_dice,tp,fp,tn,fn,sensitivity,specificity,fpr,fnr,precision,npv,accuracy,f1,Notes
0,1293,1,testing,None,0.888445,0.328682,106,152,7347,182,0.368056,0.979731,0.020269,0.631944,0.410853,0.975827,0.957108,0.388278,
1,1074,3,testing,None,0.678937,0.509165,125,76,166,34,0.786164,0.685950,0.314050,0.213836,0.621891,0.830000,0.725686,0.694444,
2,1080,3,testing,None,0.706790,0.528302,140,116,229,20,0.875000,0.663768,0.336232,0.125000,0.546875,0.919679,0.730693,0.673077,
3,2131,1,testing,None,0.866130,0.695710,519,198,1184,62,0.893287,0.856729,0.143271,0.106713,0.723849,0.950241,0.867550,0.799692,
4,1011,7,testing,None,0.765677,0.570922,161,100,232,9,0.947059,0.698795,0.301205,0.052941,0.616858,0.962656,0.782869,0.747100,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
986,1078,12,validation fold4,None,0.761905,0.000000,0,1,168,0,NaN,0.994083,0.005917,NaN,0.000000,1.000000,0.994083,NaN,
987,1050,82,validation fold4,None,0.285714,NaN,0,0,1,0,NaN,1.000000,0.000000,NaN,NaN,1.000000,1.000000,NaN,
988,1293,67,validation fold4,None,0.615385,NaN,0,0,4,0,NaN,1.000000,0.000000,NaN,NaN,1.000000,1.000000,NaN,
989,1125,96,validation fold4,None,0.602410,NaN,0,0,150,0,NaN,1.000000,0.000000,NaN,NaN,1.000000,1.000000,NaN,
